# Evidencia de ingesta OHLCV

> Cuaderno de validación reproducible del pipeline ETL de velas OHLCV (4h) sobre TimescaleDB y artefactos `parquet` en disco.

## Contexto y motivación

El núcleo cuantitativo del sistema parte de **velas OHLCV en *timeframe* 4h** para los cinco activos del universo de trading (`BTCUSDT`, `ETHUSDT`, `BNBUSDT`, `XRPUSDT`, `SOLUSDT`). Cualquier modelo predictivo o backtest aguas abajo es tan fiable como esta capa: si hay huecos, duplicados o desalineamientos temporales, el sesgo se propaga sin remedio al resto del pipeline.

La elección del cliente HTTP manual frente a `python-binance` o `ccxt` se documenta en `01_arquitectura_ingesta_datos_OHLCV.ipynb` (`notebooks/01_ingesta/ohlcv/`). Este cuaderno cierra la evidencia **operativa** del ETL productivo.

Por eso, antes de tocar *features* o modelos, conviene fijar una **evidencia objetiva y reproducible** de que la ingesta está sana. Este cuaderno materializa esa evidencia en forma de:

- una **ejecución end-to-end** del validador del pipeline (`scripts/validate_OHLCV_pipeline.py`),
- una **batería de consultas SQL** sobre las tablas `market_data.*` que cuantifica cobertura, auditoría, duplicados e imputaciones,
- un **inventario en disco** de los `parquet` raw que sirven de respaldo crudo,
- y la **exportación** de todo lo anterior como JSON/CSV versionados, de modo que la memoria del TFG pueda referenciar números trazables a una fecha de corte concreta.

El flujo está pensado para ejecutarse de una vez con `Kernel > Restart Kernel and Run All`. Cada sección añade una pieza de evidencia y termina con una breve interpretación.

## 1. Configuración del entorno

Localiza la raíz del proyecto buscando hacia arriba desde el `cwd` actual hasta encontrar `scripts/validate_OHLCV_pipeline.py`. Esto permite ejecutar el cuaderno tanto desde la carpeta `notebooks/01_ingesta/ohlcv/` como desde la raíz del repositorio o desde un contenedor Docker (donde la raíz es `/app`).

También se capturan los **metadatos de ejecución** (timestamp UTC, versión de Python y plataforma). No es decorativo: cuando alguien revise la memoria meses después necesitará poder reproducir el entorno exacto.


In [1]:
from __future__ import annotations

import contextlib
import io
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display
from sqlalchemy import text

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "scripts" / "validate_OHLCV_pipeline.py").exists():
        ROOT = candidate
        break

assert ROOT is not None, "No se encontró la raíz del proyecto (falta scripts/validate_OHLCV_pipeline.py)."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.validate_OHLCV_pipeline as val
from src.config.settings import Settings

SYMBOLS_EVIDENCIA = None
OHLCV_TABLE = "market_data.market_ohlcv"
OHLCV_OUTPUT_DIR = ROOT / "reports" / "validation" / "ohlcv"
RUN_TIMESTAMP = datetime.now(timezone.utc)
RUN_STAMP = RUN_TIMESTAMP.strftime("%Y%m%dT%H%M%SZ")

settings = Settings.load_from_yaml(ROOT / "config" / "settings.yaml")
OHLCV_FREEZE_COVERAGE_DATE = settings.trading_universe.freeze_date
symbols_evidencia = (
    tuple(symbol.upper() for symbol in SYMBOLS_EVIDENCIA)
    if SYMBOLS_EVIDENCIA
    else tuple(symbol.upper() for symbol in settings.trading_universe.symbols)
)
raw_data_dir = Path(settings.market_data.download_dir.raw_data_dir)

run_meta = {
    "fecha_hora_utc": RUN_TIMESTAMP.isoformat(),
    "raiz_proyecto": str(ROOT),
    "python": platform.python_version(),
    "plataforma": platform.platform(),
}
run_meta

/usr/local/lib/python3.10/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


{'fecha_hora_utc': '2026-06-03T23:50:27.959541+00:00',
 'raiz_proyecto': '/app',
 'python': '3.10.19',
 'plataforma': 'Linux-5.15.133.1-microsoft-standard-WSL2-x86_64-with-glibc2.36'}

## 2. Ejecución del validador end-to-end

En esta celda se invoca el validador `scripts/validate_OHLCV_pipeline.py` reutilizando exactamente la misma lógica que la *pipeline* de CI, pero adaptada al contexto interactivo del cuaderno:

- Se redirige `stdout`/`stderr` a un buffer en la propia celda para no saturar la salida del cuaderno con el logging técnico del validador.
- Si la conexión a TimescaleDB falla, todas las comprobaciones aguas abajo se marcan como `SKIP` en lugar de lanzar una excepción, dejando un informe parcial pero coherente.
- Si las tablas `market_data.*` están listas, se ejecutan en cadena: integración del pipeline -> persistencia en BD -> auditoría (`ingestion_runs`/`ingestion_events`) -> trazabilidad MLflow -> cobertura freeze (`FREEZE_COVERAGE`) contra el bucket **2026-02-28 20:00 UTC**.
- Finalmente se construye la **matriz de aceptación común** (`build_common_acceptance_matrix`) que define qué comprobaciones son obligatorias para considerar el ETL aceptado.

El resultado es la tabla `summary_df`, donde cada fila es una comprobación con su estado (`PASS`/`FAIL`/`SKIP`), si es requerida y un detalle legible. Esta tabla es el semáforo de alto nivel del ETL.


In [2]:
pipeline_result = None


def marcar_checks_dependientes_como_skip() -> None:
    val.set_status(val.results, check_id="INTEGRATION", status=val.SKIP, aliases=("PIPELINE_EXEC",))
    val.results["DB_DATA"] = val.SKIP
    val.results["DB_CASING"] = val.SKIP
    val.set_status(val.results, check_id="DB_INTEGRATION", status=val.SKIP)
    val.set_status(val.results, check_id="AUDIT_RUNS", status=val.SKIP)
    val.set_status(val.results, check_id="AUDIT_EVENTS", status=val.SKIP, aliases=("AUDIT_EVIDENCE",))
    val.set_status(val.results, check_id="DATA_QUALITY_MIN", status=val.SKIP)
    val.results["MLFLOW"] = val.SKIP
    val.results["FREEZE_COVERAGE"] = val.SKIP


# Captura el logging técnico estructurado y mantiene la salida del cuaderno limpia
technical_logs_buffer = io.StringIO()
with contextlib.redirect_stdout(technical_logs_buffer), contextlib.redirect_stderr(technical_logs_buffer):
    val.results.clear()
    pipeline_config = val.load_test_config()

    val.check_interval_to_timedelta()

    engine = val.create_db_engine()
    db_ready = bool(engine and val.test_connection(engine))

    if not db_ready:
        val.results["DB_CONNECTION"] = val.FAIL
        val.set_status(val.results, check_id="TABLES_READY", status=val.SKIP)
        marcar_checks_dependientes_como_skip()
    else:
        val.results["DB_CONNECTION"] = val.PASS
        val.check_tables_ready(engine)

        if val.results.get("TABLES_READY") == val.PASS:
            pipeline_result = val.check_pipeline_execution(pipeline_config, engine)
            val.check_db_data(engine)
            run_id = pipeline_result.get("run_id") if pipeline_result else None
            val.check_audit_tables(engine, run_id)
            val.derive_db_integration_status()
            val.check_mlflow_run()
            val.check_freeze_coverage(
                engine,
                symbols=pipeline_config["trading_universe"]["symbols"],
                target_date=None,
            )
        else:
            marcar_checks_dependientes_como_skip()

    acceptance_matrix = val.build_common_acceptance_matrix(
        target_date=val.OHLCV_FREEZE_COVERAGE_DATE,
        db_ready=db_ready,
        tables_ready=val.results.get("TABLES_READY") == val.PASS,
        data_quality_required=True,
    )
    accepted, actionable_failures = val._print_summary(acceptance_matrix)

_ = technical_logs_buffer.getvalue()
print("Ejecución completada. Se ocultó el log técnico detallado.")

matrix_required = {str(row["check_id"]): bool(row["required"]) for row in acceptance_matrix}


def detalle_check(check_id: str, status: str) -> str:
    if status == val.FAIL:
        for failure in actionable_failures:
            if check_id in failure:
                return failure
        return "Fallo detectado. Revisar logs de validación para diagnóstico."

    if status == val.SKIP:
        return "No ejecutado por precondiciones no cumplidas."

    detalles_ok = {
        "INTEGRATION": "Ejecución de integración validada correctamente.",
        "DB_INTEGRATION": "Persistencia y auditoría en BD validadas correctamente.",
        "AUDIT_RUNS": "Se validó evidencia de ejecuciones en tabla de auditoría.",
        "AUDIT_EVENTS": "Se validó evidencia de eventos asociados a ejecuciones.",
        "FREEZE_COVERAGE": (
            f"Datos en tabla con max(timestamp) >= bucket freeze 20:00 UTC del {OHLCV_FREEZE_COVERAGE_DATE} "
            f"(FREEZE_COVERAGE / OHLCV_FREEZE_COVERAGE_DATE)."
        ),
        "DB_DATA": "Se confirmaron filas persistidas para símbolos objetivo.",
        "DB_CASING": "No se detectaron violaciones de mayúsculas y minúsculas.",
        "MLFLOW": "Se verificó trazabilidad de ejecución y métricas en MLflow.",
    }
    return detalles_ok.get(check_id, "Comprobación validada correctamente.")


summary_rows = []
for check_id, label in val._CHECK_LABELS.items():
    status = val.results.get(check_id, val.SKIP)
    summary_rows.append(
        {
            "id_check": check_id,
            "etiqueta": label,
            "estado": status,
            "requerido": "Sí" if matrix_required.get(check_id, False) else "No",
            "detalle": detalle_check(check_id, status),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

Ejecución completada. Se ocultó el log técnico detallado.


,id_check,etiqueta,estado,requerido,detalle
0,DB_CONNECTION,Conexión a TimescaleDB,PASS,Sí,Comprobación validada correctamente.
1,TABLES_READY,Tablas market_data disponibles,PASS,Sí,Comprobación validada correctamente.
2,INTEGRATION,Ejecución del pipeline de integración OHLCV,PASS,Sí,Ejecución de integración validada correctamente.
3,DB_INTEGRATION,Persistencia + auditoría en BD real,PASS,Sí,Persistencia y auditoría en BD validadas corre...
4,AUDIT_RUNS,Evidencia en market_data.ingestion_runs,PASS,Sí,Se validó evidencia de ejecuciones en tabla de...
5,AUDIT_EVENTS,Evidencia en market_data.ingestion_events,PASS,Sí,Se validó evidencia de eventos asociados a eje...
6,DATA_QUALITY_MIN,Presencia de datos y formato mínimo de mayúscu...,PASS,Sí,Comprobación validada correctamente.
7,FREEZE_COVERAGE,Cobertura OHLCV por símbolo hasta bucket objet...,PASS,Sí,Datos en tabla con max(timestamp) >= bucket fr...
8,INTERVAL_HELPER,Utilidad interval_to_timedelta(),PASS,No,Comprobación validada correctamente.
9,DB_DATA,Datos insertados en market_ohlcv,PASS,No,Se confirmaron filas persistidas para símbolos...


### 2.1. Lectura de la matriz de aceptación

Las doce comprobaciones devueltas cubren las tres capas críticas del ETL:

| Capa | Comprobaciones asociadas | Por qué importa |
|---|---|---|
| **Infraestructura** | `DB_CONNECTION`, `TABLES_READY` | Sin BD viva ni tablas creadas, no hay nada que validar |
| **Pipeline** | `INTEGRATION`, `INTERVAL_HELPER` | Confirma que el código de ingesta corre de extremo a extremo sin excepciones |
| **Datos persistidos** | `DB_INTEGRATION`, `DB_DATA`, `DB_CASING`, `DATA_QUALITY_MIN` | Hay filas en `market_ohlcv`, los símbolos están normalizados a mayúsculas y el formato mínimo se respeta |
| **Auditoría** | `AUDIT_RUNS`, `AUDIT_EVENTS`, `MLFLOW` | Cada ejecución queda trazada en `ingestion_runs`/`ingestion_events` y reflejada en MLflow |
| **Cobertura temporal** | `FREEZE_COVERAGE` | Para cada símbolo, `max(timestamp) >=` **2026-02-28 20:00 UTC** (*freeze* de política); datos posteriores o iguales a ese bucket -> PASS |

**En esta ejecución** (`stamp` `20260603T234504Z`, `fecha_hora_utc` **2026-06-03 23:45 UTC**), las doce comprobaciones figuran en `PASS` y el export cierra con resultado global `PASS`. Eso valida la **base mínima** sobre la que pueden empezar a construirse *features* y modelos. Si alguna apareciera en `FAIL`, la columna `detalle` apunta a la pista accionable concreta (símbolo huérfano, tabla vacía, ejecución sin métrica MLflow, etc.).


## 3. Evidencia SQL: cobertura, auditoría e integridad

En esta sección se lanzan **cuatro** consultas SQL contra `market_data.*`:

1. **Cobertura por símbolo y timeframe** sobre `market_ohlcv` -> filas, primer y último timestamp.
2. **Ejecuciones de ingesta OHLCV (resumen por modo):** tabla **agregada** sobre `ingestion_runs` para los `pipeline_type` (`etl_ohlcv`, `etl_ohlcv_live`, `etl_ohlcv_backfill`): cuenta de ejecuciones y métricas por modo **histórico vs. live**; única evidencia de auditoría de runs en este cuaderno (agregado global, sin listado run a run)
3. **Duplicados por PK** `(symbol, timeframe, timestamp)` debe ser estrictamente cero.
4. **Integridad real vs. imputaciones** mediante el flag `is_imputed`, expresado como porcentaje.

Adicionalmente se construye un **inventario en disco** de los `parquet` raw `<SIMBOLO>_4h.parquet` bajo `market_data.download_dir.raw_data_dir`. Sirve como respaldo crudo independiente de la BD: si algún día hay que reconstruir la tabla, el Parquet es la fuente de verdad.

> Nota técnica: **`max_timestamp`** es el máximo real en BD por símbolo (el live puede desalinear series entre sí). El cumplimiento del freeze de política (`FREEZE_COVERAGE`) va aparte, en el validador end-to-end.


In [3]:
resumen_simbolos_df = pd.DataFrame(
    columns=["simbolo", "tabla", "filas", "min_timestamp", "max_timestamp"]
)
resumen_modos_df = pd.DataFrame()

_OHLCV_PT_SQL = "'etl_ohlcv', 'etl_ohlcv_live', 'etl_ohlcv_backfill'"

if db_ready and engine:
    cobertura_query = text(
        f"""
        SELECT
            m.symbol,
            m.timeframe,
            COUNT(*) AS filas,
            MIN(m.timestamp) AS min_timestamp,
            MAX(m.timestamp) AS max_timestamp
        FROM {OHLCV_TABLE} m
        WHERE m.timeframe = '4h'
        GROUP BY m.symbol, m.timeframe
        ORDER BY m.symbol;
        """
    )
    cobertura = pd.DataFrame({"simbolo": list(symbols_evidencia)}).merge(
        pd.read_sql(cobertura_query, engine)
        .rename(columns={"symbol": "simbolo"})
        [["simbolo", "filas", "min_timestamp", "max_timestamp"]],
        on="simbolo",
        how="left",
    )
    cobertura["filas"] = cobertura["filas"].fillna(0).astype(int)
    cobertura["max_timestamp"] = pd.to_datetime(
        cobertura["max_timestamp"], utc=True, errors="coerce"
    )
    resumen_simbolos_df = cobertura.assign(tabla=OHLCV_TABLE)[
        ["simbolo", "tabla", "filas", "min_timestamp", "max_timestamp"]
    ]

    # Resumen agregado por modo sobre ingestion_runs
    resumen_modos_query = text(
        f"""
        SELECT
            CASE
                WHEN LOWER(CAST(r.pipeline_type AS TEXT)) LIKE '%live%' THEN 'live'
                ELSE 'histórico'
            END AS modo,
            COUNT(*)::bigint AS ejecuciones,
            MAX(r.started_at) AS ultimo_inicio,
            COALESCE(SUM(r.total_rows), 0)::bigint AS total_filas,
            COALESCE(SUM(r.failed_count), 0)::bigint AS total_fallos_eventos,
            SUM(CASE WHEN COALESCE(r.failed_count, 0) > 0 THEN 1 ELSE 0 END)::bigint AS ejecuciones_con_fallo
        FROM market_data.ingestion_runs r
        WHERE r.pipeline_type IN ({_OHLCV_PT_SQL})
        GROUP BY 1
        ORDER BY modo;
        """
    )
    resumen_modos_df = pd.read_sql(resumen_modos_query, engine)
    if not resumen_modos_df.empty:
        resumen_modos_df["pct_ejecuciones_con_fallo"] = (
            (resumen_modos_df["ejecuciones_con_fallo"] / resumen_modos_df["ejecuciones"]) * 100.0
        ).round(2)

# Calcula la evidencia de duplicados por PK
duplicados_pk_df = pd.DataFrame()
if db_ready and engine:
    duplicados_query = text(
        f"""
        SELECT symbol, timeframe,
               COUNT(*)                       AS total_filas,
               COUNT(DISTINCT timestamp)      AS timestamps_unicos,
               COUNT(*) - COUNT(DISTINCT timestamp) AS duplicados_pk
        FROM {OHLCV_TABLE}
        WHERE timeframe = '4h'
        GROUP BY symbol, timeframe
        ORDER BY symbol;
        """
    )
    duplicados_pk_df = pd.read_sql(duplicados_query, engine).rename(columns={"symbol": "simbolo"})

# Calcula la evidencia de integridad e imputaciones por símbolo
integridad_df = pd.DataFrame()
if db_ready and engine:
    integridad_query = text(
        f"""
        SELECT symbol,
               COUNT(*)                                                AS total_filas,
               SUM(CASE WHEN is_imputed THEN 1 ELSE 0 END)            AS filas_imputadas,
               SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END)        AS filas_reales,
               ROUND(
                   100.0 * SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END)
                   / NULLIF(COUNT(*), 0), 4
               )                                                       AS pct_integridad_real
        FROM {OHLCV_TABLE}
        WHERE timeframe = '4h'
        GROUP BY symbol
        ORDER BY symbol;
        """
    )
    integridad_df = pd.read_sql(integridad_query, engine).rename(columns={"symbol": "simbolo"})

# Construye el inventario de artefactos Parquet en disco
parquet_rows: list[dict] = []
for symbol in symbols_evidencia:
    parquet_path = raw_data_dir / f"{symbol}_4h.parquet"
    if parquet_path.exists():
        parquet_size_mb = parquet_path.stat().st_size / (1024 * 1024)
        try:
            parquet_df = pd.read_parquet(parquet_path)
            parquet_timestamps = (
                pd.to_datetime(parquet_df["timestamp"], utc=True, errors="coerce").dropna()
                if "timestamp" in parquet_df.columns
                else pd.Series(dtype="datetime64[ns, UTC]")
            )
            parquet_rows.append(
                {
                    "tipo": "raw_ohlcv",
                    "simbolo": symbol,
                    "archivo": parquet_path.name,
                    "tamano_mb": round(parquet_size_mb, 2),
                    "filas": len(parquet_df),
                    "columnas": len(parquet_df.columns),
                    "min_timestamp": parquet_timestamps.min() if not parquet_timestamps.empty else None,
                    "max_timestamp": parquet_timestamps.max() if not parquet_timestamps.empty else None,
                }
            )
        except Exception:
            parquet_rows.append(
                {
                    "tipo": "raw_ohlcv",
                    "simbolo": symbol,
                    "archivo": parquet_path.name,
                    "tamano_mb": round(parquet_size_mb, 2),
                    "filas": None,
                    "columnas": None,
                    "min_timestamp": None,
                    "max_timestamp": None,
                }
            )
    else:
        parquet_rows.append(
            {
                "tipo": "raw_ohlcv",
                "simbolo": symbol,
                "archivo": f"{symbol}_4h.parquet",
                "tamano_mb": None,
                "filas": None,
                "columnas": None,
                "min_timestamp": None,
                "max_timestamp": None,
            }
        )

parquet_inventory_df = pd.DataFrame(parquet_rows)

## 4. Resultados y análisis

A continuación se muestran en orden: cobertura OHLCV, **resumen agregado de ejecuciones** por modo en `ingestion_runs`, duplicados PK, integridad/imputaciones y el inventario Parquet, con **el estado completo del ETL** a la fecha de corte.


In [4]:
print("Resumen de ejecuciones OHLCV por modo (ingestion_runs, agregado global)")
display(resumen_modos_df)

print("\nCobertura por símbolo (tabla origen)")
display(resumen_simbolos_df)

print("\nDuplicados por PK (símbolo, timeframe, timestamp)")
display(duplicados_pk_df)
if not duplicados_pk_df.empty and (duplicados_pk_df["duplicados_pk"] == 0).all():
    print("Cero duplicados confirmados para todos los símbolos.")
else:
    print("Se detectaron duplicados o la consulta no devolvió resultados.")

print("\nIntegridad e imputaciones por símbolo")
display(integridad_df)

print("\nInventario de artefactos Parquet en disco")
display(parquet_inventory_df)

Resumen de ejecuciones OHLCV por modo (ingestion_runs, agregado global)


,modo,ejecuciones,ultimo_inicio,total_filas,total_fallos_eventos,ejecuciones_con_fallo,pct_ejecuciones_con_fallo
0,histórico,94,2026-06-03 23:50:29.467543+00:00,85770,0,0,0.00
1,live,144,2026-05-31 03:20:37.510789+00:00,5461,5,5,3.47



Cobertura por símbolo (tabla origen)


,simbolo,tabla,filas,min_timestamp,max_timestamp
0,BTCUSDT,market_data.market_ohlcv,19277,2017-08-17 04:00:00+00:00,2026-06-03 20:00:00+00:00
1,ETHUSDT,market_data.market_ohlcv,19254,2017-08-17 04:00:00+00:00,2026-05-31 00:00:00+00:00
2,BNBUSDT,market_data.market_ohlcv,18769,2017-11-06 00:00:00+00:00,2026-05-31 00:00:00+00:00
3,XRPUSDT,market_data.market_ohlcv,17693,2018-05-04 08:00:00+00:00,2026-05-31 00:00:00+00:00
4,SOLUSDT,market_data.market_ohlcv,12714,2020-08-11 04:00:00+00:00,2026-05-31 00:00:00+00:00



Duplicados por PK (símbolo, timeframe, timestamp)


,simbolo,timeframe,total_filas,timestamps_unicos,duplicados_pk
0,BNBUSDT,4h,18769,18769,0
1,BTCUSDT,4h,19277,19277,0
2,ETHUSDT,4h,19254,19254,0
3,SOLUSDT,4h,12714,12714,0
4,XRPUSDT,4h,17693,17693,0


Cero duplicados confirmados para todos los símbolos.

Integridad e imputaciones por símbolo


,simbolo,total_filas,filas_imputadas,filas_reales,pct_integridad_real
0,BNBUSDT,18769,0,18769,100.0
1,BTCUSDT,19277,0,19277,100.0
2,ETHUSDT,19254,0,19254,100.0
3,SOLUSDT,12714,0,12714,100.0
4,XRPUSDT,17693,0,17693,100.0



Inventario de artefactos Parquet en disco


,tipo,simbolo,archivo,tamano_mb,filas,columnas,min_timestamp,max_timestamp
0,raw_ohlcv,BTCUSDT,BTCUSDT_4h.parquet,1.16,19277,11,2017-08-17 04:00:00+00:00,2026-06-03 20:00:00+00:00
1,raw_ohlcv,ETHUSDT,ETHUSDT_4h.parquet,1.06,18765,11,2017-08-17 04:00:00+00:00,2026-03-10 12:00:00+00:00
2,raw_ohlcv,BNBUSDT,BNBUSDT_4h.parquet,0.94,18280,11,2017-11-06 00:00:00+00:00,2026-03-10 12:00:00+00:00
3,raw_ohlcv,XRPUSDT,XRPUSDT_4h.parquet,0.92,17204,11,2018-05-04 08:00:00+00:00,2026-03-10 12:00:00+00:00
4,raw_ohlcv,SOLUSDT,SOLUSDT_4h.parquet,0.62,12225,11,2020-08-11 04:00:00+00:00,2026-03-10 12:00:00+00:00


### 4.1. Interpretación de los resultados

**Resumen histórico/live.** En esta ejecución el cohorte suma **237** ejecuciones en `ingestion_runs`: **93** en modo histórico (**39.2 %**) y **144** en live (**60.8 %**). Histórico: último inicio **2026-06-03 23:45 UTC**, **85,770** filas acumuladas reportadas, **0** fallos de evento y **0 %** de ejecuciones con fallo. Live: último inicio **2026-05-31 03:20 UTC**, **5,461** filas reportadas, **5** fallos de evento acumulados y **3.47 %** de ejecuciones con al menos un fallo (**5** corridas). El modo live concentra la mayoría de las ejecuciones registradas; los fallos de evento son residuales frente al volumen total.

**Cobertura por símbolo.** El cumplimiento formal del **freeze** (`FREEZE_COVERAGE`) es independiente de esta tabla; aquí **`max_timestamp`** es el máximo real en BD por símbolo (el live puede desalinear series). Los **`min_timestamp`** reflejan el listado en Binance y marcan la **ventana mínima** común para modelos multivariantes:

- `BTCUSDT` / `ETHUSDT`: ~ ago 2017 (**19,277** / **19,254** velas 4h en esta corrida); `BTCUSDT` alcanza **2026-06-03 20:00 UTC** en `max_timestamp`.
- `BNBUSDT`: ~ nov 2017 (**18,769** velas); `ETHUSDT`/`BNBUSDT`/`XRPUSDT`/`SOLUSDT` con `max_timestamp` **2026-05-31 00:00 UTC** en esta corrida.
- `XRPUSDT`: ~ may 2018 (**17,693** velas).
- `SOLUSDT`: ~ ago 2020 (**12,714** velas).

**Duplicados por PK.** `duplicados_pk = 0` y `total_filas == timestamps_unicos` validan la PK `(symbol, timeframe, timestamp)` (`docker/init.sql`) y descartan doble ingesta por reintentos (`ON CONFLICT DO UPDATE`).

**Integridad e imputaciones.** La salida muestra `pct_integridad_real = 100.0` (100 % en todos los símbolos) y `filas_imputadas = 0`.

**Inventario Parquet.** Coherencia universo configurado vs archivos en disco; respaldo para reconstruir BD sin re-pull al exchange.

## 5. Exportación de artefactos para reproducibilidad

Las tablas vistas son útiles dentro del cuaderno, pero la memoria del TFG y la auditoría externa necesitan **artefactos persistentes** en disco. En esta celda se vuelca todo lo anterior bajo `reports/validation/ohlcv/`, con un sello UTC (`<stamp>`) en el nombre para no sobrescribir corridas previas (p. ej. `20260603T234504Z` en esta corrida).

- **JSON consolidado:** `ohlcv_validation_<stamp>.json` (metadatos, `fecha_objetivo_freeze_resuelta`, `target_bucket_utc`, símbolos, resultado global, fallos accionables, `checks` y bloques equivalentes a los CSV).

- **Seis CSV** (mismo `<stamp>`): `ohlcv_validation_summary_`, `resumen_simbolos_`, `modos_`, `duplicados_`, `integridad_`, `parquet_inventory_`.

- **Figuras:** ninguna en este cuaderno (solo tablas y exportación tabular).

Al final se imprime el **resultado global** (`PASS`/`FAIL`) y un mini-checklist (histórico, *live*, duplicados, integridad). Por último se libera la conexión con `engine.dispose()` si el kernel sigue vivo.


In [5]:
# Exporta la evidencia consolidada
OHLCV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_paths = {
    "json": OHLCV_OUTPUT_DIR / f"ohlcv_validation_{RUN_STAMP}.json",
    "summary": OHLCV_OUTPUT_DIR / f"ohlcv_validation_summary_{RUN_STAMP}.csv",
    "resumen_simbolos": OHLCV_OUTPUT_DIR / f"ohlcv_validation_resumen_simbolos_{RUN_STAMP}.csv",
    "modos": OHLCV_OUTPUT_DIR / f"ohlcv_validation_modos_{RUN_STAMP}.csv",
    "duplicados": OHLCV_OUTPUT_DIR / f"ohlcv_validation_duplicados_{RUN_STAMP}.csv",
    "integridad": OHLCV_OUTPUT_DIR / f"ohlcv_validation_integridad_{RUN_STAMP}.csv",
    "parquet_inventory": OHLCV_OUTPUT_DIR / f"ohlcv_validation_parquet_inventory_{RUN_STAMP}.csv",
}

historico_ok = bool((not resumen_modos_df.empty) and (resumen_modos_df["modo"] == "histórico").any())
live_ok = bool((not resumen_modos_df.empty) and (resumen_modos_df["modo"] == "live").any())
duplicados_ok = bool(not duplicados_pk_df.empty and (duplicados_pk_df["duplicados_pk"] == 0).all())
integridad_ok = bool(not integridad_df.empty and len(integridad_df) == len(symbols_evidencia))

payload = {
    "metadata": run_meta,
    "fecha_objetivo_freeze_resuelta": OHLCV_FREEZE_COVERAGE_DATE,
    "target_bucket_utc": val._target_bucket_start(OHLCV_FREEZE_COVERAGE_DATE).isoformat(),
    "symbols_evidencia": list(symbols_evidencia),
    "resultado_global": "PASS" if accepted else "FAIL",
    "fallos_accionables": actionable_failures,
    "checks": summary_df.to_dict(orient="records"),
    "resumen_simbolos": resumen_simbolos_df.to_dict(orient="records") if not resumen_simbolos_df.empty else [],
    "modos": resumen_modos_df.to_dict(orient="records") if not resumen_modos_df.empty else [],
    "completitud_etl": {
        "historico_detectado": historico_ok,
        "live_detectado": live_ok,
    },
    "duplicados_pk": {
        "verificado": duplicados_ok,
        "cero_duplicados_todos_simbolos": duplicados_ok,
        "detalle": duplicados_pk_df.to_dict(orient="records") if not duplicados_pk_df.empty else [],
    },
    "integridad_imputaciones": {
        "verificado": integridad_ok,
        "detalle": integridad_df.to_dict(orient="records") if not integridad_df.empty else [],
    },
    "inventario_parquet": {
        "archivos_encontrados": int((parquet_inventory_df["tamano_mb"].notna()).sum()) if not parquet_inventory_df.empty else 0,
        "archivos_esperados": len(parquet_inventory_df) if not parquet_inventory_df.empty else 0,
        "detalle": parquet_inventory_df.to_dict(orient="records") if not parquet_inventory_df.empty else [],
    },
}

export_paths["json"].write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
summary_df.to_csv(export_paths["summary"], index=False)
if not resumen_simbolos_df.empty:
    resumen_simbolos_df.to_csv(export_paths["resumen_simbolos"], index=False)
if not resumen_modos_df.empty:
    resumen_modos_df.to_csv(export_paths["modos"], index=False)
if not duplicados_pk_df.empty:
    duplicados_pk_df.to_csv(export_paths["duplicados"], index=False)
if not integridad_df.empty:
    integridad_df.to_csv(export_paths["integridad"], index=False)
if not parquet_inventory_df.empty:
    parquet_inventory_df.to_csv(export_paths["parquet_inventory"], index=False)

print(f"Resultado global: {'PASS' if accepted else 'FAIL'}")
print(f"Completitud histórico: {'OK' if historico_ok else 'PENDIENTE'}")
print(f"Completitud live: {'OK' if live_ok else 'PENDIENTE'}")
print(f"Duplicados = 0 verificado: {'OK' if duplicados_ok else 'PENDIENTE'}")
print(f"Integridad por símbolo: {'OK' if integridad_ok else 'PENDIENTE'}")
print(f"Evidencia JSON: {export_paths['json']}")
print(f"Resumen de comprobaciones CSV: {export_paths['summary']}")
if not resumen_simbolos_df.empty:
    print(f"Resumen por símbolo CSV: {export_paths['resumen_simbolos']}")
if not resumen_modos_df.empty:
    print(f"Resumen ejecuciones por modo (ingestion_runs) CSV: {export_paths['modos']}")
if not duplicados_pk_df.empty:
    print(f"Duplicados PK CSV: {export_paths['duplicados']}")
if not integridad_df.empty:
    print(f"Integridad/imputaciones CSV: {export_paths['integridad']}")
if not parquet_inventory_df.empty:
    print(f"Inventario Parquet CSV: {export_paths['parquet_inventory']}")

if engine:
    engine.dispose()


Resultado global: PASS
Completitud histórico: OK
Completitud live: OK
Duplicados = 0 verificado: OK
Integridad por símbolo: OK
Evidencia JSON: /app/reports/validation/ohlcv/ohlcv_validation_20260603T235027Z.json
Resumen de comprobaciones CSV: /app/reports/validation/ohlcv/ohlcv_validation_summary_20260603T235027Z.csv
Resumen por símbolo CSV: /app/reports/validation/ohlcv/ohlcv_validation_resumen_simbolos_20260603T235027Z.csv
Resumen ejecuciones por modo (ingestion_runs) CSV: /app/reports/validation/ohlcv/ohlcv_validation_modos_20260603T235027Z.csv
Duplicados PK CSV: /app/reports/validation/ohlcv/ohlcv_validation_duplicados_20260603T235027Z.csv
Integridad/imputaciones CSV: /app/reports/validation/ohlcv/ohlcv_validation_integridad_20260603T235027Z.csv
Inventario Parquet CSV: /app/reports/validation/ohlcv/ohlcv_validation_parquet_inventory_20260603T235027Z.csv


## 6. Conclusiones

Para el universo de cinco símbolos en *timeframe* 4h, esta ejecución (`stamp` `20260603T234504Z`, `fecha_hora_utc` **2026-06-03 23:45 UTC**) deja fijada una lectura en tres capas:

1. **Validador:** resultado global `PASS`; las doce comprobaciones de `summary_df` en `PASS` (incluido `FREEZE_COVERAGE` hasta **2026-02-28 20:00 UTC**).
2. **SQL y auditoría:** **87,707** velas 4h agregadas en cobertura por símbolo; **0** duplicados PK; integridad real al **100 %** sin imputaciones; cohorte de **237** ejecuciones (`ingestion_runs`) con histórico y *live* detectados.
3. **Reproducibilidad:** JSON + seis CSV bajo `reports/validation/ohlcv/`; inventario *Parquet* raw coherente con el universo configurado.

**Nota.** Resultado global, estados de `summary_df`, métricas SQL y rutas con `stamp` dependen del estado del proyecto al ejecutar el cuaderno; en otra corrida pueden cambiar.